# V-RAG Evaluation

Evaluates any of the three models on the entity-probing benchmark, with retrieval enabled or
disabled, and reports the metrics used in the paper. One configuration is run per execution, so
the full comparison is produced by repeating the notebook across model and mode.

## Inputs
- `chexpert_cache/` (image shards + manifest) - required in every mode
- `vrag_database.zip` - required only when retrieval is used
- CheXzero weights - required only for similarity-based retrieval, downloaded automatically
- `test_vqa.jsonl` - the benchmark (6,940 questions over 1,000 images, balanced Yes/No)
- The model: the base repository, or a LoRA adapter archive for LLaVA-S or LLaVA-Vrag

## Retrieval modes
- `off` - the query image and the question only; no retrieval, and neither CheXzero nor the
  database is loaded
- `vrag` - CheXzero embeds the query, FAISS returns the three most similar cases, and their
  images and reports are supplied as references
- `random` - three randomly chosen cases are supplied instead. This control separates the benefit
  of relevant evidence from the benefit of simply having extra context

## Outputs (all tagged with model and mode)
- `eval_<label>.log` - milestones, progress and the final report
- `report_<label>.txt` and `metrics_<label>.json` - the metrics
- `preds_<label>.jsonl` - per-question predictions, allowing a run to resume
- `eval_dataset_<label>.jsonl` and `dataset_preview_<label>.pdf` - the exact model inputs
- `results_grid.csv` - the comparison across all completed runs

## How hallucination is measured
Free-text hallucination is difficult to score objectively, so each finding is probed
individually. A false positive is the model asserting a finding that is not present, so
precision corresponds to the hallucination rate, and MCC summarises overall agreement with the
ground truth without being inflated by a model that answers one way.

## Configuration

The only cell edited between runs. Selects the model, the retrieval mode, and the input paths.
Two flags are derived from the mode and control whether the database and CheXzero are loaded at
all, so a no-retrieval run touches neither.

In [ ]:
# ============================================================
#  CONFIG  — the only cell you edit between runs
# ============================================================
MODEL    = "llava_s"          # "llava" | "llava_s" | "llava_vrag"
RAG_MODE = "vrag"             # "off" | "vrag" | "random"
LIMIT    = None               # None = all 6,940 rows; int = quick pass

# ---- model registry: name -> where it comes from ----
#   base    : an HF repo id Unsloth loads directly (no adapter)
#   adapter : a .zip (or a dir) holding adapter_config.json + adapter weights
MODELS = {
    "llava":      {"kind": "base",    "ref": "unsloth/llava-1.5-7b-hf-bnb-4bit"},
    "llava_s":    {"kind": "adapter", "ref": "/workspace/llava_s_adapter.zip"},
    "llava_vrag": {"kind": "adapter", "ref": "/workspace/llava_vrag_adapter.zip"},
}

# ---- inputs you place on the pod (edit these paths) ----
CACHE_DIR     = "/workspace/chexpert_cache"     # images_part*.zip + manifest.csv     (ALL modes)
VRAG_DB_ZIP   = "/workspace/vrag_database.zip"  # builder output; auto-extracted      (vrag/random)
DATABASE_DIR  = "/workspace/vrag_database"      # extract target (index.faiss + parquet)
CHEXZERO_DIR  = "/workspace/chexzero"           # CheXzero *.pt; AUTO-downloaded       (vrag only)
TEST_VQA_PATH = "/workspace/test_vqa.jsonl"     # you upload                          (ALL modes)
OUT_DIR       = "/workspace/vrag_eval_out"
CHEXZERO_DRIVE = "https://drive.google.com/drive/folders/1makFLiEMbSleYltaRxw81aBhEDMpVwno"

# ---- retrieval / budget / generation ----
TOP_K          = 3
MAX_SEQ_LEN    = 4096
MAX_NEW_TOKENS = 4
# rows per model.generate() call. off is light (1 small image) -> big batch; vrag/random carry
# 4 images + ~2800 tokens -> smaller. Set 1 to disable batching. A guard in the sanity cell verifies
# batched==single before the sweep trusts it. If vrag OOMs, lower this.
BATCH_SIZE     = 16 if RAG_MODE == "off" else 4
CHEXZERO_BATCH = 256
LOG_EVERY      = 200          # rows between progress log lines (background-safe)
N_PREVIEW      = 20           # datapoints rendered into the dataset-preview PDF
SEED           = 42

# ---- validation thresholds (irregularity detection) ----
VALIDATE_IMAGES = True        # open every used image, flag missing/blank/undecodable
BLANK_STD       = 1.0         # grayscale std below this = "blank/near-constant" image
LOW_SIM_WARN    = 0.5         # retrieval top-sim below this = "weak retrieval"

assert MODEL in MODELS, f"unknown MODEL {MODEL!r}; choose from {list(MODELS)}"
assert RAG_MODE in ("off", "vrag", "random")
LABEL         = f"{MODEL}_{RAG_MODE}"
NEED_DATABASE = RAG_MODE in ("vrag", "random")   # random needs reports/keys; off needs nothing
NEED_CHEXZERO = RAG_MODE == "vrag"               # only cosine retrieval embeds the query
print(f"LABEL={LABEL} | {MODELS[MODEL]} | DB={NEED_DATABASE} CheXzero={NEED_CHEXZERO} | LIMIT={LIMIT}")

## Imports and logging

Sets up logging to both the notebook and a file, with the run label on every line. The log file
is what makes a long background run observable: it records progress, warnings and the final
report even if the interface disconnects.

In [ ]:
import os, io, re, json, csv, time, glob, random, hashlib, zipfile, logging, sys, math
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np, pandas as pd
from PIL import Image
import torch
from tqdm.auto import tqdm

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

# ---- logging: writes to eval_{LABEL}.log AND the notebook (label on every line) ----
LOG_PATH = Path(OUT_DIR) / f"eval_{LABEL}.log"
logger = logging.getLogger("vrag_eval"); logger.setLevel(logging.INFO); logger.handlers.clear()
_fmt = logging.Formatter(f"%(asctime)s | {LABEL} | %(message)s", "%H:%M:%S")
for _h in (logging.FileHandler(LOG_PATH), logging.StreamHandler(sys.stdout)):
    _h.setFormatter(_fmt); logger.addHandler(_h)
def log(msg): logger.info(msg)

log(f"===== EVAL START | MODEL={MODEL} RAG_MODE={RAG_MODE} LIMIT={LIMIT} =====")
log(f"device: {device} | {torch.cuda.get_device_name(0) if device=='cuda' else 'cpu-only'} | log -> {LOG_PATH}")

## Prepare the inputs

Resolves the input paths and extracts the retrieval database if it is still archived. The
database and encoder are only prepared when the selected mode requires them. Missing inputs fail
here with an explicit message rather than midway through the run.

In [ ]:
def dir_containing(root, pattern):
    """directory that (directly, or nested) contains files matching pattern; assert-or-raise."""
    root = Path(root)
    if list(root.glob(pattern)):
        return root
    for p in root.rglob(pattern):
        return p.parent
    raise AssertionError(f"'{pattern}' not found under {root} — did you place/extract it?")

# cache + test_vqa: needed in EVERY mode
CACHE = dir_containing(CACHE_DIR, "images_part*")
assert Path(TEST_VQA_PATH).exists(), f"test_vqa not found: {TEST_VQA_PATH}"

DB_ROOT = CHEXZERO = None

# database: only vrag/random. Extract vrag_database.zip once if not already extracted.
if NEED_DATABASE:
    if not list(Path(DATABASE_DIR).rglob("database.parquet")):
        assert Path(VRAG_DB_ZIP).exists(), f"neither extracted DB nor {VRAG_DB_ZIP} present"
        log(f"extracting {VRAG_DB_ZIP} -> {DATABASE_DIR}")
        Path(DATABASE_DIR).mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(VRAG_DB_ZIP) as zf:
            for m in tqdm([n for n in zf.namelist() if ":Zone.Identifier" not in n], desc="extract db"):
                zf.extract(m, DATABASE_DIR)
    DB_ROOT = dir_containing(DATABASE_DIR, "database.parquet")

# CheXzero: only vrag. Auto-download from Drive if the .pt isn't there.
if NEED_CHEXZERO:
    if not list(Path(CHEXZERO_DIR).rglob("*.pt")):
        import gdown
        log(f"CheXzero: downloading from Drive -> {CHEXZERO_DIR}")
        Path(CHEXZERO_DIR).mkdir(parents=True, exist_ok=True)
        gdown.download_folder(CHEXZERO_DRIVE, output=CHEXZERO_DIR, quiet=False, use_cookies=False)
    CHEXZERO = dir_containing(CHEXZERO_DIR, "*.pt")

log(f"resolved | CACHE={CACHE} | DB_ROOT={DB_ROOT} | CHEXZERO={CHEXZERO} | TEST_VQA={TEST_VQA_PATH}")

## Index the image cache

Builds the mapping from image key to shard so images can be read directly from the compressed
shards, and defines a health check that flags images which are missing, unreadable or blank.

In [ ]:
parts = sorted(CACHE.glob("images_part*"))
extracted = bool(parts) and parts[0].is_dir()
man = CACHE / "manifest.csv"
key2src = {}
if man.exists():
    total = sum(1 for _ in open(man)) - 1
    for row in tqdm(csv.reader(open(man)), total=max(total, 0), desc="cache index"):
        if not row or row[0] in ("key", "png_key"):
            continue
        k, sh = row[0], row[1]
        key2src[k] = ("file", str(CACHE / sh.replace(".zip", "") / k)) if extracted else ("zip", str(CACHE / sh))
else:
    for p in tqdm(parts, desc="scan zips"):
        if zipfile.is_zipfile(p):
            with zipfile.ZipFile(p) as zf:
                for n in zf.namelist():
                    if n.endswith(".png"):
                        key2src[n] = ("zip", str(p))
assert key2src, "no images under CACHE"
log(f"cache index: {len(key2src):,} images ({'dirs' if extracted else 'zips'})")

_zh = {}
def read_image(key):
    kind, loc = key2src[key]
    if kind == "file":
        return Image.open(loc).convert("RGB")
    if loc not in _zh:
        _zh[loc] = zipfile.ZipFile(loc)
    return Image.open(io.BytesIO(_zh[loc].read(key))).convert("RGB")

def image_health(key):
    """'ok' | reason. Catches missing / undecodable / zero-size / blank(near-constant)."""
    if key not in key2src:
        return "missing"
    try:
        im = read_image(key)
    except Exception as e:
        return f"decode:{type(e).__name__}"
    if im.size[0] == 0 or im.size[1] == 0:
        return "zero-size"
    if float(np.asarray(im.convert("L"), dtype=np.float32).std()) < BLANK_STD:
        return "blank"
    return "ok"

## Load the benchmark

Loads the benchmark questions and confirms every referenced image is present in the cache.

In [ ]:
def to_png_key(p):
    parts = list(Path(str(p).strip()).parts)
    while parts and (parts[0] in ("train", "valid", "test") or parts[0].startswith("CheXpert")):
        parts = parts[1:]
    return str(Path(*parts).with_suffix(".png"))

rows = [json.loads(l) for l in open(TEST_VQA_PATH) if l.strip()]
for r in tqdm(rows, desc="prep test_vqa"):
    r["png_key"] = to_png_key(r["path"])
df_vqa = pd.DataFrame(rows)

log(f"test_vqa: {len(df_vqa):,} rows | {df_vqa.png_key.nunique():,} images | "
    f"balance {dict(Counter(df_vqa.answer))}")
miss = [k for k in df_vqa.png_key.unique() if k not in key2src]
if miss:
    log(f"WARN {len(miss)} query images NOT in cache, e.g. {miss[:3]}")
    assert len(miss) < len(df_vqa.png_key.unique()), "NO query images resolve — wrong cache?"
else:
    log("all query images present in cache")
df_vqa[["vqa_id", "png_key", "entity", "answer"]].head()

## Load CheXzero

Loads the retrieval encoder and verifies it reproduces identical embeddings for the same image.
Preprocessing matches the database build exactly, which is required for query and database
vectors to be comparable. This cell is skipped entirely unless similarity retrieval is used.

In [ ]:
if NEED_CHEXZERO:
    import clip
    from torchvision.transforms import Resize, Normalize, InterpolationMode

    cz_files = sorted(glob.glob(str(Path(CHEXZERO) / "**" / "*.pt"), recursive=True), key=os.path.getsize)
    assert cz_files, f"no .pt under {CHEXZERO}"
    sd = torch.load(cz_files[0], map_location="cpu"); sd = sd.get("state_dict", sd)
    sd = {k.replace("module.", ""): v for k, v in sd.items()}
    cz_model = clip.model.build_model(sd).to(device).eval()
    CZ_RES = cz_model.visual.input_resolution
    CZ_DIM = cz_model.text_projection.shape[1]

    _rz = Resize(CZ_RES, interpolation=InterpolationMode.BICUBIC, antialias=True)
    _nm = Normalize([101.48761] * 3, [83.43944] * 3)     # CheXzero: 0-255 scale, NO /255
    def cz_tf(pil):
        a = np.asarray(pil.convert("RGB")).astype(np.float32)
        return _nm(_rz(torch.from_numpy(a).permute(2, 0, 1)))

    @torch.no_grad()
    def cz_embed(keys, desc="chexzero"):
        out = np.zeros((len(keys), CZ_DIM), dtype=np.float32)
        for s in tqdm(range(0, len(keys), CHEXZERO_BATCH), desc=desc):
            batch = keys[s:s + CHEXZERO_BATCH]
            px = torch.stack([cz_tf(read_image(k)) for k in batch]).to(device)
            f = torch.nn.functional.normalize(cz_model.encode_image(px).float(), dim=-1)
            out[s:s + len(batch)] = f.cpu().numpy()
        return out

    _k0 = df_vqa.png_key.iloc[0]
    _a = cz_embed([_k0], desc="verify"); _b = cz_embed([_k0], desc="verify")
    assert np.isfinite(_a).all(), "NaN/inf in CheXzero output"
    _sim = float((_a[0] * _b[0]).sum())
    assert _sim > 0.999, f"CheXzero not deterministic (self-sim {_sim:.3f})"
    log(f"CheXzero ready: dim={CZ_DIM} res={CZ_RES} | self-sim {_sim:.4f}")
else:
    log(f"CheXzero: skipped (RAG_MODE={RAG_MODE} needs no cosine retrieval)")

## Retrieve references

Embeds every unique query image and searches the FAISS index for the most similar cases,
discarding any neighbour belonging to the query's own patient. The random-control variant draws
its references from the same database without using similarity. The encoder is released from
memory afterwards so it does not compete with the model for GPU memory.

In [ ]:
retr = {}                       # query_key -> top-k refs (vrag only)
db_key = db_report = db_patient = None

if NEED_DATABASE:
    db = pd.read_parquet(Path(DB_ROOT) / "database.parquet")
    db_key     = db["png_key"].values
    db_report  = db["report"].values
    db_patient = db["deid_patient_id"].astype(str).values
    log(f"DB: {len(db):,} rows loaded")

if RAG_MODE == "vrag":
    import faiss
    index = faiss.read_index(str(Path(DB_ROOT) / "index.faiss"))
    assert index.ntotal == len(db), f"index {index.ntotal} != parquet {len(db)}"
    uniq_keys = [k for k in df_vqa.png_key.unique() if k in key2src]
    q_emb = cz_embed(uniq_keys, desc="embed queries").astype(np.float32)
    BUF = TOP_K + 6
    sims, ids = index.search(q_emb, BUF)
    skipped = 0
    for i, qk in enumerate(tqdm(uniq_keys, desc="assemble top-k")):
        qpat = qk.split("/")[0]; refs = []
        for j, s in zip(ids[i], sims[i]):
            j = int(j)
            if db_patient[j] == qpat or db_key[j] == qk:      # no self-leak
                skipped += 1; continue
            refs.append({"png_key": db_key[j], "report": str(db_report[j]), "sim": float(s)})
            if len(refs) == TOP_K: break
        retr[qk] = refs
    assert all(len(v) == TOP_K for v in retr.values()), "a query is short of TOP_K after self-leak filter"
    log(f"retrieved top-{TOP_K} for {len(retr):,} images | self-hits skipped {skipped} | "
        f"median top-1 sim {np.median(sims[:, 0]):.3f}")
    del cz_model; torch.cuda.empty_cache(); log("CheXzero freed")

def _rand_refs(qk):
    qpat = qk.split("/")[0]
    rr = random.Random(int(hashlib.md5(qk.encode()).hexdigest()[:8], 16))
    out = []
    while len(out) < TOP_K:
        j = rr.randrange(len(db_key))
        if db_patient[j] == qpat: continue
        out.append({"png_key": db_key[j], "report": str(db_report[j]), "sim": None})
    return out

def refs_for(qk, mode):
    if mode == "off":    return []
    if mode == "vrag":   return retr.get(qk, [])
    if mode == "random": return _rand_refs(qk)
    raise ValueError(mode)

## Build the evaluation dataset

Assembles the exact input for every benchmark question according to the selected mode, then
validates it and saves it. Validation counts blank or missing images, empty reference reports,
weak retrievals and any mismatch between image placeholders and images, and reports them as a
summary so problems are visible in the log rather than silently affecting the scores. Samples
are printed so the constructed input can be read directly.

In [ ]:
ORD = ["1st", "2nd", "3rd", "4th", "5th"]

def build_prompt(entity, refs):
    e = entity.strip().lower()
    if not refs:
        return f"<image>\nAnswer with only the word yes or no. Does the patient have {e}?", 1
    parts = []
    for i, r in enumerate(refs):
        parts.append(f"<image>\nThis is the {ORD[i]} similar image and its report "
                     f"for your reference. {r['report']}")
    parts.append("Answer the question with only the word yes or no. Do not provide explanations. "
                 "According to the last query image and the reference images and reports, does the "
                 f"patient have {e}?")
    parts.append("<image>")                          # query image LAST
    return "\n".join(parts), len(refs) + 1

dataset, irr = [], Counter()
for r in tqdm(df_vqa.itertuples(index=False), total=len(df_vqa), desc=f"build {RAG_MODE}"):
    if r.png_key not in key2src:
        irr["query_missing_from_cache"] += 1
    refs = refs_for(r.png_key, RAG_MODE)
    prompt, n_img = build_prompt(r.entity, refs)
    image_keys = ([x["png_key"] for x in refs] + [r.png_key]) if refs else [r.png_key]
    if not (prompt.count("<image>") == len(image_keys) == n_img):
        irr["prompt_image_mismatch"] += 1
    if len(set(image_keys)) != len(image_keys):
        irr["duplicate_images_in_prompt"] += 1
    for x in refs:
        if not str(x["report"]).strip():
            irr["empty_ref_report"] += 1
        if x["sim"] is not None and x["sim"] < LOW_SIM_WARN:
            irr["weak_retrieval_top<%.1f" % LOW_SIM_WARN] += 1
    est_tok = n_img * 576 + int(len(prompt.replace("<image>", "").split()) * 1.4) + MAX_NEW_TOKENS
    if est_tok > MAX_SEQ_LEN:
        irr["est_tokens>MAX_SEQ_LEN"] += 1
    dataset.append({"vqa_id": r.vqa_id, "entity": r.entity, "gt": r.answer,
                    "mode": RAG_MODE, "query_key": r.png_key,
                    "ref_keys": [x["png_key"] for x in refs],
                    "ref_reports": [str(x["report"]) for x in refs],
                    "ref_sims": [x["sim"] for x in refs],
                    "image_keys": image_keys, "prompt": prompt, "n_img": n_img})

if VALIDATE_IMAGES:
    uniq = sorted({k for dp in dataset for k in dp["image_keys"]})
    bad = {}
    for k in tqdm(uniq, desc="validate images"):
        h = image_health(k)
        if h != "ok":
            bad[k] = h; irr["img_" + h.split(":")[0]] += 1
    for k, h in list(bad.items())[:10]:
        log(f"WARN bad image [{h}]: {k}")
    log(f"validated {len(uniq):,} unique images | bad {len(bad)}")

ds_path = Path(OUT_DIR) / f"eval_dataset_{LABEL}.jsonl"
with open(ds_path, "w") as f:
    for dp in tqdm(dataset, desc="save dataset"):
        f.write(json.dumps(dp) + "\n")
log(f"built dataset: {len(dataset):,} datapoints [{RAG_MODE}] -> {ds_path}")
log("IRREGULARITY SUMMARY (build): " + (json.dumps(dict(irr)) if irr else "none"))

for dp in dataset[:2]:
    log("--- sample datapoint ---")
    log(f"vqa_id={dp['vqa_id']} entity={dp['entity']} gt={dp['gt']} "
        f"n_img={dp['n_img']} query={dp['query_key']}")
    for i, (k, rep, s) in enumerate(zip(dp["ref_keys"], dp["ref_reports"], dp["ref_sims"])):
        log(f"  ref{i+1} sim={s} {k} :: {rep[:90].strip()}...")
print("\nSAMPLE PROMPT (datapoint 0):\n" + "-" * 70)
print(dataset[0]["prompt"][:1400] + ("..." if len(dataset[0]["prompt"]) > 1400 else ""))

## Render the inputs to PDF

Renders the first samples as one page per question: the images in the order they are fed, each
labelled as a reference or the query and marked with its health status, followed by the input as
an interleaved sequence. This makes the model input inspectable rather than implied.

In [ ]:
import textwrap
import matplotlib.pyplot as plt
from matplotlib.backends.backend_pdf import PdfPages

def _feed_sequence(dp):
    """the interleaved model input as text: image slots + text segments, in feed order."""
    pieces, lines, img_i = dp["prompt"].split("<image>"), [], 0
    for j, part in enumerate(pieces):
        txt = part.strip()
        if txt:
            for ln in textwrap.wrap(txt, 116)[:14]:
                lines.append("      " + ln)
        if j < dp["n_img"]:
            k = dp["image_keys"][img_i]
            label = "QUERY" if k == dp["query_key"] else f"ref{img_i + 1}"
            lines.append(f"  >>> [ IMAGE {img_i + 1} : {label} ]  {k}")
            img_i += 1
    return "\n".join(lines)

def _render_page(dp, pdf):
    keys = dp["image_keys"]; n = len(keys)
    fig = plt.figure(figsize=(8.5, 11))
    for idx, k in enumerate(keys):                       # images row (feed order)
        ax = fig.add_axes([0.03 + idx * (0.94 / n), 0.72, (0.94 / n) * 0.9, 0.20])
        h = image_health(k)
        try: ax.imshow(read_image(k), cmap="gray")
        except Exception: ax.text(0.5, 0.5, "UNREADABLE", ha="center", va="center")
        ax.axis("off")
        lbl = "QUERY" if k == dp["query_key"] else f"ref{idx + 1}"
        ax.set_title(f"[{idx + 1}] {lbl}\n{h}", fontsize=7, color=("green" if h == "ok" else "red"))
    ax2 = fig.add_axes([0.03, 0.02, 0.94, 0.66]); ax2.axis("off")   # feed transcript
    header = (f"vqa_id={dp['vqa_id']}   entity={dp['entity']}   GT={dp['gt']}   "
              f"mode={dp['mode']}   images fed = {n}")
    body = header + "\n" + "=" * 92 + "\nMODEL INPUT (exact feed order — each IMAGE slot is encoded):\n\n" + _feed_sequence(dp)
    ax2.text(0, 1, body, fontsize=6.5, family="monospace", va="top")
    pdf.savefig(fig); plt.close(fig)

pdf_path = Path(OUT_DIR) / f"dataset_preview_{LABEL}.pdf"
nprev = min(N_PREVIEW, len(dataset))
with PdfPages(pdf_path) as pdf:
    for dp in tqdm(dataset[:nprev], desc="render pdf"):
        _render_page(dp, pdf)
log(f"dataset preview PDF ({nprev} pages) -> {pdf_path}")

## Load the model

Resolves the selected model to a base repository or an adapter directory, extracting the adapter
archive if required, and loads it for inference.

In [ ]:
from unsloth import FastVisionModel

def resolve_model(name):
    spec = MODELS[name]
    if spec["kind"] == "base":
        log(f"model '{name}': base repo -> {spec['ref']}")
        return spec["ref"]
    ref = Path(spec["ref"])
    assert ref.exists(), f"adapter for '{name}' not found: {ref}"
    if ref.suffix == ".zip":
        dst = Path(OUT_DIR) / f"_model_{name}"
        if not list(dst.rglob("adapter_config.json")):
            log(f"model '{name}': unzipping {ref.name} -> {dst}")
            with zipfile.ZipFile(ref) as zf:
                members = [n for n in zf.namelist() if ":Zone.Identifier" not in n and not n.endswith("/")]
                for m in tqdm(members, desc="unzip adapter"):
                    zf.extract(m, dst)
        search_root = dst
    else:
        search_root = ref
    cfgs = list(Path(search_root).rglob("adapter_config.json"))
    assert cfgs, f"no adapter_config.json under {search_root}"
    adir = str(cfgs[0].parent)
    log(f"model '{name}': adapter dir -> {adir}")
    return adir

MODEL_SRC = resolve_model(MODEL)
log(f"loading {MODEL_SRC} (4-bit, ~1-2 min silent) ...")
model, tokenizer = FastVisionModel.from_pretrained(MODEL_SRC, load_in_4bit=True)
FastVisionModel.for_inference(model)
try: model.max_seq_length = MAX_SEQ_LEN
except Exception: pass
for _t in (tokenizer, getattr(tokenizer, "tokenizer", None)):
    if _t is not None:
        try: _t.model_max_length = MAX_SEQ_LEN
        except Exception: pass
        try: _t.padding_side = "left"           # decoder-only: LEFT-pad for correct batched generation
        except Exception: pass
log(f"model ready (4-bit inference) | BATCH_SIZE={BATCH_SIZE} | padding_side=left")

## Inference

Builds the input for one question and returns the parsed Yes/No answer, the raw output and the
input length. Batched inference is also defined, with padding on the left so that generated
tokens begin at the same position for every row in a batch.

In [ ]:
def parse_yes_no(text):
    t = text.strip().lower()
    if t.startswith("yes"): return "yes"
    if t.startswith("no"):  return "no"
    head = t.split()[:3]
    if "yes" in head: return "yes"
    if "no" in head:  return "no"
    return "no"

def _processor(imgs, text):
    """Handle both transformers signatures without guessing: keyword first, positional fallback."""
    try:
        return tokenizer(images=imgs, text=text, add_special_tokens=False, return_tensors="pt")
    except TypeError:
        return tokenizer(imgs, text, add_special_tokens=False, return_tensors="pt")

@torch.no_grad()
def infer(dp):
    imgs = [read_image(k) for k in dp["image_keys"]]
    prompt = dp["prompt"]
    assert prompt.count("<image>") == len(imgs) == dp["n_img"], \
        f"{dp['vqa_id']}: <image>={prompt.count('<image>')} imgs={len(imgs)} n={dp['n_img']}"
    pieces, content = prompt.split("<image>"), []
    for j, part in enumerate(pieces):
        if part.strip(): content.append({"type": "text", "text": part})
        if j < len(imgs): content.append({"type": "image", "image": imgs[j]})
    messages = [{"role": "user", "content": content}]
    text = tokenizer.apply_chat_template(messages, add_generation_prompt=True)
    inputs = _processor(imgs, text).to(device)
    tok_len = int(inputs["input_ids"].shape[1])
    out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, use_cache=True)
    gen = tokenizer.decode(out[0][tok_len:], skip_special_tokens=True)
    return parse_yes_no(gen), gen.strip(), tok_len

@torch.no_grad()
def infer_batch(dps):
    """Batched generation. Relies on LEFT padding (set at model load) so every row's generated
    tokens start at the same position L. Returns [(pred, raw, tok_len), ...] aligned to dps."""
    imgs_list, texts = [], []
    for dp in dps:
        imgs = [read_image(k) for k in dp["image_keys"]]
        assert dp["prompt"].count("<image>") == len(imgs) == dp["n_img"], f"{dp['vqa_id']}: image count"
        pieces, content = dp["prompt"].split("<image>"), []
        for j, part in enumerate(pieces):
            if part.strip(): content.append({"type": "text", "text": part})
            if j < len(imgs): content.append({"type": "image", "image": imgs[j]})
        texts.append(tokenizer.apply_chat_template([{"role": "user", "content": content}],
                                                   add_generation_prompt=True))
        imgs_list.append(imgs)
    try:
        inputs = tokenizer(images=imgs_list, text=texts, padding=True,
                           add_special_tokens=False, return_tensors="pt").to(device)
    except TypeError:
        inputs = tokenizer(imgs_list, texts, padding=True,
                           add_special_tokens=False, return_tensors="pt").to(device)
    L = inputs["input_ids"].shape[1]
    out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, use_cache=True)
    res = []
    for i in range(len(dps)):
        gen = tokenizer.decode(out[i][L:], skip_special_tokens=True)   # left-pad -> gen at end
        res.append((parse_yes_no(gen), gen.strip(), L))
    return res

log("inference helpers ready")

## Sanity check and timing

Runs a small number of questions and prints the model's raw output next to the ground truth. It
also confirms that the images actually reach the model by inspecting the processed image tensor,
verifies that batched inference agrees with single-row inference before the full run relies on
it, and measures the true per-question time so the total runtime is known in advance.

In [ ]:
# ---- IMAGE-FEED CHECK: prove pixel_values actually reach the model ----
_dp = dataset[0]
_imgs = [read_image(k) for k in _dp["image_keys"]]
_pieces, _content = _dp["prompt"].split("<image>"), []
for j, part in enumerate(_pieces):
    if part.strip(): _content.append({"type": "text", "text": part})
    if j < len(_imgs): _content.append({"type": "image", "image": _imgs[j]})
_txt = tokenizer.apply_chat_template([{"role": "user", "content": _content}], add_generation_prompt=True)
_inp = _processor(_imgs, _txt)
_pv = _inp.get("pixel_values", None)
if _pv is None:
    log("IMAGE-FEED CHECK: NO pixel_values in inputs — images may NOT be reaching the model!")
else:
    okimg = _pv.shape[0] == len(_imgs)
    log(f"IMAGE-FEED CHECK: n_img={_dp['n_img']} | pixel_values={tuple(_pv.shape)} | "
        f"input_ids={tuple(_inp['input_ids'].shape)} -> {'all images fed' if okimg else 'shape[0]≠n_img, check layout'}")

# ---- BATCH GUARD: batched inference MUST match single-row, or padding/alignment is broken ----
if BATCH_SIZE > 1:
    _n = min(BATCH_SIZE, 8, len(dataset))
    _single  = [infer(dp)[0] for dp in dataset[:_n]]
    _batched = [r[0] for r in infer_batch(dataset[:_n])]
    _agree = sum(a == b for a, b in zip(_single, _batched))
    log(f"BATCH GUARD: batched vs single agree {_agree}/{_n}")
    if _agree != _n:
        log(f"BATCH GUARD FAILED (single={_single} batched={_batched}) — batching is unreliable on "
            f"this build; FALLING BACK to BATCH_SIZE=1 (sequential, correct, slower).")
        BATCH_SIZE = 1
    else:
        log("BATCH GUARD — sweep will run batched")

times = []
log("--- sanity (10 datapoints) ---")
for dp in tqdm(dataset[:10], desc="sanity"):
    t0 = time.time(); pred, raw, tok = infer(dp); dt = time.time() - t0
    times.append(dt)
    ok = "OK " if pred == str(dp["gt"]).strip().lower() else "MISS"
    flag = "" if raw.lower().startswith(("yes", "no")) else "  <<AMBIGUOUS"
    log(f"  [{ok}] {dp['entity']:22s} tok={tok} GT={dp['gt']:3s} "
        f"PRED={pred:3s} raw={raw!r}{flag} ({dt:.2f}s)")
steady = float(np.median(times[2:])) if len(times) > 2 else float(np.mean(times))
n = LIMIT if LIMIT is not None else len(dataset)
log(f"median {steady:.2f}s/row -> full {n:,} rows @ {RAG_MODE} ~ {steady * n / 3600:.1f} h on this GPU")

## Run the full evaluation

Runs every question, writing each prediction as it is produced so an interrupted run resumes
where it stopped. Progress is logged at intervals with a running accuracy, and unparseable
outputs and inference errors are counted and reported rather than being discarded silently.

In [ ]:
pred_path = Path(OUT_DIR) / f"preds_{LABEL}.jsonl"
done = set()
if pred_path.exists():
    for l in open(pred_path):
        try: done.add(json.loads(l)["vqa_id"])
        except Exception: pass
    log(f"resuming — {len(done):,} rows already saved")

work = dataset if LIMIT is None else dataset[:LIMIT]
todo = [dp for dp in work if dp["vqa_id"] not in done]
log(f"sweep start: {len(todo):,} / {len(work):,} rows -> {pred_path}")

def _chunks(lst, n):
    for i in range(0, len(lst), n): yield lst[i:i + n]

correct = ambiguous = errors = overlong = seen_ct = 0; t0 = time.time()
with open(pred_path, "a") as f:
    for batch in tqdm(list(_chunks(todo, BATCH_SIZE)), desc=f"{LABEL} (bs={BATCH_SIZE})"):
        try:
            results = infer_batch(batch)                    # batched (or size-1) forward
        except Exception as e:                              # OOM/other -> isolate rows, don't lose the batch
            log(f"WARN batch failed ({type(e).__name__}: {str(e)[:80]}) — per-row fallback")
            results = []
            for dp in batch:
                try: results.append(infer(dp))
                except Exception as e2:
                    errors += 1; log(f"WARN infer-error {dp['vqa_id']}: {type(e2).__name__}")
                    results.append(("no", f"ERROR:{type(e2).__name__}", 0))
        for dp, (pred, raw, tok) in zip(batch, results):
            seen_ct += 1
            if not raw.strip().lower().startswith(("yes", "no")):
                ambiguous += 1
                if ambiguous <= 20: log(f"WARN ambiguous {dp['vqa_id']}: {raw!r}")
            if tok and tok > MAX_SEQ_LEN: overlong += 1
            if pred == str(dp["gt"]).strip().lower(): correct += 1
            f.write(json.dumps({"vqa_id": dp["vqa_id"], "entity": dp["entity"],
                                "gt": dp["gt"], "pred": pred, "raw": raw, "tok_len": tok}) + "\n")
        f.flush()
        if seen_ct % LOG_EVERY < BATCH_SIZE or seen_ct == len(todo):
            el = time.time() - t0; rate = seen_ct / el if el else 0
            eta = (len(todo) - seen_ct) / rate if rate else 0
            log(f"progress {seen_ct}/{len(todo)} ({seen_ct/len(todo)*100:.0f}%) | running acc {correct/seen_ct:.3f} | "
                f"{rate:.2f} rows/s | elapsed {el/60:.1f}m | eta {eta/60:.1f}m")
log(f"IRREGULARITY SUMMARY (sweep): ambiguous {ambiguous} | infer-errors {errors} | overlong {overlong}")
log(f"sweep done -> {pred_path}")

## Compute the metrics

Computes accuracy, precision, recall, F1 and MCC from the saved predictions and writes the report
to the log, to a report file and to a machine-readable file. Precision is the hallucination
measure; MCC is reported because on a balanced Yes/No task F1 can be high for a model that
simply answers one way, whereas MCC cannot.

In [ ]:
recs = [json.loads(l) for l in open(pred_path)]

def binm(rs):
    yes = lambda v: str(v).strip().lower() == "yes"
    tp = sum(1 for r in rs if yes(r["gt"]) and yes(r["pred"]))
    tn = sum(1 for r in rs if not yes(r["gt"]) and not yes(r["pred"]))
    fp = sum(1 for r in rs if not yes(r["gt"]) and yes(r["pred"]))
    fn = sum(1 for r in rs if yes(r["gt"]) and not yes(r["pred"]))
    n = len(rs); acc = (tp + tn) / n if n else 0.0
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    den = math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn)) or 1.0
    mcc = (tp * tn - fp * fn) / den
    return dict(n=n, acc=acc, precision=prec, recall=rec, f1=f1, mcc=mcc, tp=tp, tn=tn, fp=fp, fn=fn)

overall = binm(recs)

report = []
def rep(s): report.append(s); log(s)
rep("================ REPORT " + LABEL + " ================")
rep(f"MODEL={MODEL}  RAG_MODE={RAG_MODE}  n={overall['n']}")
rep(f"  overall  acc={overall['acc']:.3f}  F1={overall['f1']:.3f}  "
    f"P={overall['precision']:.3f}  R={overall['recall']:.3f}  MCC={overall['mcc']:+.3f}")
by_ent = defaultdict(list)
for r in recs: by_ent[r["entity"]].append(r)
rep("  per-entity (n>=20):")
for ent, rs in sorted(by_ent.items(), key=lambda kv: -len(kv[1])):
    if len(rs) < 20: continue
    m = binm(rs)
    rep(f"    {ent:28s} n={m['n']:4d} acc={m['acc']:.3f} F1={m['f1']:.3f} MCC={m['mcc']:+.3f}")
rep("=" * (25 + len(LABEL)))

(Path(OUT_DIR) / f"report_{LABEL}.txt").write_text("\n".join(report))
json.dump({"model": MODEL, "rag_mode": RAG_MODE, "overall": overall},
          open(Path(OUT_DIR) / f"metrics_{LABEL}.json", "w"), indent=2)
log(f"report -> report_{LABEL}.txt | metrics -> metrics_{LABEL}.json | log -> eval_{LABEL}.log")

## Qualitative comparison

Displays the images and references for individual questions with the model's answer, and, when
both the retrieval and no-retrieval runs exist for a model, reports how many answers retrieval
corrected and how many it broke.

In [ ]:
import matplotlib.pyplot as plt

for dp in dataset[:3]:
    keys = dp["image_keys"]
    fig, axes = plt.subplots(1, len(keys), figsize=(3.6 * len(keys), 3.6))
    if len(keys) == 1: axes = [axes]
    for ax, k in zip(axes, keys):
        ax.imshow(read_image(k)); ax.axis("off")
        ax.set_title("QUERY" if k == dp["query_key"] else "ref", fontsize=9)
    pred, _, _ = infer(dp)
    fig.suptitle(f"[{RAG_MODE}] {dp['entity']} | GT={dp['gt']} PRED={pred}", fontsize=11)
    plt.tight_layout(); plt.show()
    for i, rep_ in enumerate(dp["ref_reports"]):
        print(f"  ref{i+1}: {rep_[:180].strip()}...")

off_p  = Path(OUT_DIR) / f"preds_{MODEL}_off.jsonl"
vrag_p = Path(OUT_DIR) / f"preds_{MODEL}_vrag.jsonl"
if off_p.exists() and vrag_p.exists():
    off = {json.loads(l)["vqa_id"]: json.loads(l) for l in open(off_p)}
    vr  = {json.loads(l)["vqa_id"]: json.loads(l) for l in open(vrag_p)}
    common = set(off) & set(vr); y = lambda v: str(v).strip().lower()
    fixed = [i for i in common if y(off[i]["pred"]) != y(off[i]["gt"]) and y(vr[i]["pred"]) == y(vr[i]["gt"])]
    broke = [i for i in common if y(off[i]["pred"]) == y(off[i]["gt"]) and y(vr[i]["pred"]) != y(vr[i]["gt"])]
    print(f"\ncommon {len(common):,} | V-RAG fixed {len(fixed):,} | broke {len(broke):,} | net {len(fixed)-len(broke):+d}")
    for i in fixed[:5]:
        print(f"  {i}: {off[i]['entity']} GT={off[i]['gt']} off={off[i]['pred']} -> vrag={vr[i]['pred']}")
else:
    print("\n(run RAG_MODE=off and =vrag for this model to enable the on-vs-off diff)")

## Results grid

Collects every completed run into a single table of model against retrieval mode, which is the
comparison reported in the paper.

In [ ]:
grid_rows = []
for mp in sorted(glob.glob(str(Path(OUT_DIR) / "metrics_*.json"))):
    d = json.load(open(mp))
    m = d["overall"]
    grid_rows.append({"model": d["model"], "mode": d["rag_mode"],
                      "n": m["n"], "acc": round(m["acc"], 3), "f1": round(m["f1"], 3),
                      "mcc": round(m["mcc"], 3)})
if grid_rows:
    grid = pd.DataFrame(grid_rows)
    grid.to_csv(Path(OUT_DIR) / "results_grid.csv", index=False)
    for metric in ("acc", "mcc", "f1"):
        print(f"\n=== overall {metric} ===")
        print(grid.pivot_table(index="model", columns="mode", values=metric).to_string())
    print(f"\nsaved -> {Path(OUT_DIR) / 'results_grid.csv'}")
else:
    print("no metrics_*.json yet — run some (MODEL, RAG_MODE) combos first")

## Reproducing the full comparison

Run the notebook once per combination of model and retrieval mode. Outputs are tagged by
combination so runs do not overwrite each other, and the final cell assembles them into the
comparison table.